# Bellabeat Case Study — Data Cleaning & Exploratory Analysis (Python)

## Project Links & Deliverables
* **Full Case Study Report (Google Doc):** [Bellabeat Case Study - Analysis Report](https://docs.google.com/document/d/1IcYpZHLALhgdnSn3PaQAjSBy5Jkhu1lDwTbtAn85g4A/edit?usp=sharing)
* **Interactive Dashboard (Tableau Public):** [Bellabeat Case Study Dashboard on Tableau Public](https://public.tableau.com:9443/views/BeallabeatXaseStudyActivitySedentaryTimeSleepPatterns/BellabeatCaseStudyActivitySedentaryTimeSleepPatterns?:language=fr-FR&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)
* **Code & Project Repository (GitHub):** [Bellabeat Case Study GitHub Repository](https://github.com/Roseline-HM/Bellabeat-Case-Study)

## Overview & Objective
This Kaggle notebook documents the data audit, cleaning, feature engineering, and exploratory analysis for the **Bellabeat** marketing analytics case study using the public **FitBit Fitness Tracker Data** (`Fitabase Data 4.12.16-5.12.16`, April 12 – May 12, 2016). The goal is to uncover daily movement, sedentary, and sleep patterns to guide recommendations for the **Bellabeat app** and **Bellabeat membership**.

## Key Data Preparation & Findings
* **Data Cleaning & Consolidation:** Standardized timestamps, removed **3 duplicate rows** from `sleep_day` (leaving **410 clean rows**), audited missingness (including `Fat` in weight logs and **77 zero-step days** in daily activity), engineered helper fields (`DayOfWeek`, `Hour`, `TotalActiveMinutes`, `TotalHoursAsleep`, `SleepEfficiency`), and exported clean datasets (`hourly_summary_export.csv`, `activity_sleep_merged.csv`, `daily_activity_clean.csv`, `sleep_day_clean.csv`, `user_summary_export.csv`)[cite: 13].
* **Core Takeaways:** Users averaged **7,637.91 daily steps** (over 78% below 10,000 steps) and **991.21 sedentary minutes** (~16.52 hours/day), while averaging **6.99 hours** of sleep with a negative correlation between sedentary time and sleep duration (`r = -0.601`)[cite: 13]. Activity peaked on **Saturdays** and **Tuesdays** and during **17:00–19:00** (highest at **18:00**), whereas **Sunday** showed the lowest average steps (`6,933.23`) and longest sleep (`452.75` minutes / `7.55` hours)[cite: 13].

For the full step-by-step walkthrough across **Ask, Prepare, Process, Analyze, Share, and Act**, see the Google Doc report linked above[cite: 13].

In [1]:
import pandas as pd

# 1. Define the exact directory path in Kaggle
base_path = "/kaggle/input/datasets/arashnic/fitbit/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16"

# 2. Load the five selected datasets
daily_activity = pd.read_csv(f"{base_path}/dailyActivity_merged.csv")
sleep_day = pd.read_csv(f"{base_path}/sleepDay_merged.csv")
hourly_steps = pd.read_csv(f"{base_path}/hourlySteps_merged.csv")
hourly_intensities = pd.read_csv(f"{base_path}/hourlyIntensities_merged.csv")
weight_log = pd.read_csv(f"{base_path}/weightLogInfo_merged.csv")

# 3. Initial data audit across all five tables
tables = {
    "daily_activity": daily_activity,
    "sleep_day": sleep_day,
    "hourly_steps": hourly_steps,
    "hourly_intensities": hourly_intensities,
    "weight_log": weight_log
}

for name, df in tables.items():
    print(f"--- {name.upper()} ---")
    print(f"Shape (rows, columns) : {df.shape}")
    print(f"Unique users (Id)     : {df['Id'].nunique()}")
    print(f"Exact duplicates      : {df.duplicated().sum()}")
    print(f"Missing values (NaN)  : {df.isna().sum().sum()}\n")

--- DAILY_ACTIVITY ---
Shape (rows, columns) : (940, 15)
Unique users (Id)     : 33
Exact duplicates      : 0
Missing values (NaN)  : 0

--- SLEEP_DAY ---
Shape (rows, columns) : (413, 5)
Unique users (Id)     : 24
Exact duplicates      : 3
Missing values (NaN)  : 0

--- HOURLY_STEPS ---
Shape (rows, columns) : (22099, 3)
Unique users (Id)     : 33
Exact duplicates      : 0
Missing values (NaN)  : 0

--- HOURLY_INTENSITIES ---
Shape (rows, columns) : (22099, 4)
Unique users (Id)     : 33
Exact duplicates      : 0
Missing values (NaN)  : 0

--- WEIGHT_LOG ---
Shape (rows, columns) : (67, 8)
Unique users (Id)     : 8
Exact duplicates      : 0
Missing values (NaN)  : 65



In [2]:
# 1. Inspect data types and sample date formats before cleaning
for name, df in tables.items():
    print(f"--- {name.upper()} DTYPES ---")
    print(df.dtypes, "\n")

# 2. Remove exact duplicates from sleep_day (3 duplicate rows found)
sleep_day_clean = sleep_day.drop_duplicates().copy()
print(f"sleep_day rows before: {len(sleep_day)} | after removing duplicates: {len(sleep_day_clean)}\n")

# 3. Standardize date/datetime formats across tables
daily_activity_clean = daily_activity.copy()
daily_activity_clean['ActivityDate'] = pd.to_datetime(daily_activity_clean['ActivityDate'], format='%m/%d/%Y')

sleep_day_clean['SleepDay'] = pd.to_datetime(sleep_day_clean['SleepDay'], format='%m/%d/%Y %I:%M:%S %p')

hourly_steps_clean = hourly_steps.copy()
hourly_steps_clean['ActivityHour'] = pd.to_datetime(hourly_steps_clean['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')

hourly_intensities_clean = hourly_intensities.copy()
hourly_intensities_clean['ActivityHour'] = pd.to_datetime(hourly_intensities_clean['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')

weight_log_clean = weight_log.copy()
weight_log_clean['Date'] = pd.to_datetime(weight_log_clean['Date'], format='%m/%d/%Y %I:%M:%S %p')

# 4. Check for potential anomalies (e.g., days with 0 total steps or 1440 sedentary minutes = device not worn)
zero_step_days = daily_activity_clean[daily_activity_clean['TotalSteps'] == 0]
print(f"Days with 0 TotalSteps: {len(zero_step_days)} out of {len(daily_activity_clean)} rows")
print(f"Missing values per column in weight_log:\n{weight_log_clean.isna().sum()}\n")

# 5. Feature engineering: create derived metrics for analysis
daily_activity_clean['DayOfWeek'] = daily_activity_clean['ActivityDate'].dt.day_name()
daily_activity_clean['TotalActiveMinutes'] = (
    daily_activity_clean['VeryActiveMinutes'] +
    daily_activity_clean['FairlyActiveMinutes'] +
    daily_activity_clean['LightlyActiveMinutes']
)
daily_activity_clean['TotalTrackedMinutes'] = (
    daily_activity_clean['TotalActiveMinutes'] + daily_activity_clean['SedentaryMinutes']
)

sleep_day_clean['DayOfWeek'] = sleep_day_clean['SleepDay'].dt.day_name()
sleep_day_clean['TotalHoursAsleep'] = (sleep_day_clean['TotalMinutesAsleep'] / 60).round(2)
sleep_day_clean['SleepEfficiency'] = (
    (sleep_day_clean['TotalMinutesAsleep'] / sleep_day_clean['TotalTimeInBed']) * 100
).round(2)

# 6. Merge hourly_steps and hourly_intensities into a single hourly table
hourly_activity_clean = pd.merge(
    hourly_steps_clean,
    hourly_intensities_clean,
    on=['Id', 'ActivityHour'],
    how='inner'
)
hourly_activity_clean['Hour'] = hourly_activity_clean['ActivityHour'].dt.hour
hourly_activity_clean['DayOfWeek'] = hourly_activity_clean['ActivityHour'].dt.day_name()

print(f"hourly_activity_clean shape: {hourly_activity_clean.shape}")
print("Data cleaning and transformation completed successfully!")

--- DAILY_ACTIVITY DTYPES ---
Id                            int64
ActivityDate                 object
TotalSteps                    int64
TotalDistance               float64
TrackerDistance             float64
LoggedActivitiesDistance    float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
dtype: object 

--- SLEEP_DAY DTYPES ---
Id                     int64
SleepDay              object
TotalSleepRecords      int64
TotalMinutesAsleep     int64
TotalTimeInBed         int64
dtype: object 

--- HOURLY_STEPS DTYPES ---
Id               int64
ActivityHour    object
StepTotal        int64
dtype: object 

--- HOURLY_INTENSITIES DTYPES ---
Id                    int64
ActivityHour         object
TotalIntensity        int

In [3]:
import numpy as np

# 1. Descriptive statistics: daily activity
activity_cols = [
    "TotalSteps",
    "TotalDistance",
    "VeryActiveMinutes",
    "FairlyActiveMinutes",
    "LightlyActiveMinutes",
    "SedentaryMinutes",
    "TotalActiveMinutes",
    "Calories",
]
print("=== DAILY ACTIVITY SUMMARY STATS (ALL 940 ROWS) ===")
print(daily_activity_clean[activity_cols].describe().round(2), "\n")

# Descriptive statistics excluding 0-step days (likely non-wear days)
active_days = daily_activity_clean[daily_activity_clean["TotalSteps"] > 0]
print(
    f"=== DAILY ACTIVITY SUMMARY STATS (ACTIVE STEP DAYS ONLY: {len(active_days)} ROWS) ==="
)
print(
    active_days[
        ["TotalSteps", "TotalActiveMinutes", "SedentaryMinutes", "Calories"]
    ]
    .describe()
    .round(2),
    "\n",
)

# 2. Descriptive statistics: sleep
sleep_cols = [
    "TotalMinutesAsleep",
    "TotalHoursAsleep",
    "TotalTimeInBed",
    "SleepEfficiency",
]
print("=== SLEEP SUMMARY STATS (410 CLEAN ROWS) ===")
print(sleep_day_clean[sleep_cols].describe().round(2), "\n")

# 3. Trends by day of the week
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

daily_by_dow = (
    daily_activity_clean.groupby("DayOfWeek")[
        [
            "TotalSteps",
            "VeryActiveMinutes",
            "TotalActiveMinutes",
            "SedentaryMinutes",
            "Calories",
        ]
    ]
    .mean()
    .reindex(day_order)
    .round(2)
)
print("=== AVERAGE DAILY ACTIVITY BY DAY OF WEEK ===")
print(daily_by_dow, "\n")

sleep_by_dow = (
    sleep_day_clean.groupby("DayOfWeek")[
        ["TotalMinutesAsleep", "TotalHoursAsleep", "TotalTimeInBed", "SleepEfficiency"]
    ]
    .mean()
    .reindex(day_order)
    .round(2)
)
print("=== AVERAGE SLEEP BY DAY OF WEEK ===")
print(sleep_by_dow, "\n")

# 4. Hourly trends: peak activity hours
hourly_by_hour = (
    hourly_activity_clean.groupby("Hour")[
        ["StepTotal", "TotalIntensity", "AverageIntensity"]
    ]
    .mean()
    .round(2)
)
print("=== TOP 5 MOST ACTIVE HOURS (BY AVERAGE STEPS) ===")
print(hourly_by_hour.sort_values("StepTotal", ascending=False).head(5), "\n")

print("=== TOP 5 LEAST ACTIVE HOURS (BY AVERAGE STEPS) ===")
print(hourly_by_hour.sort_values("StepTotal", ascending=True).head(5), "\n")

# 5. Merge daily activity and sleep data on Id and Date
daily_activity_clean["DateKey"] = daily_activity_clean["ActivityDate"].dt.date
sleep_day_clean["DateKey"] = sleep_day_clean["SleepDay"].dt.date

activity_sleep_merged = pd.merge(
    daily_activity_clean,
    sleep_day_clean,
    on=["Id", "DateKey"],
    how="inner",
)
print(f"Merged activity + sleep rows: {len(activity_sleep_merged)}")
print("=== CORRELATION: STEPS, ACTIVE/SEDENTARY MINUTES, AND SLEEP ===")
corr_cols = [
    "TotalSteps",
    "VeryActiveMinutes",
    "SedentaryMinutes",
    "Calories",
    "TotalMinutesAsleep",
    "SleepEfficiency",
]
print(activity_sleep_merged[corr_cols].corr().round(3), "\n")

# 6. User-level engagement and activity segmentation
user_summary = (
    daily_activity_clean.groupby("Id")
    .agg(
        tracked_days=("ActivityDate", "count"),
        avg_steps=("TotalSteps", "mean"),
        avg_sedentary_min=("SedentaryMinutes", "mean"),
        avg_very_active_min=("VeryActiveMinutes", "mean"),
        avg_calories=("Calories", "mean"),
    )
    .round(1)
)


def classify_user(steps):
    if steps < 5000:
        return "Sedentary (<5k steps)"
    elif steps < 7500:
        return "Low Active (5k-7.4k steps)"
    elif steps < 10000:
        return "Somewhat Active (7.5k-9.9k steps)"
    else:
        return "Active (10k+ steps)"


user_summary["ActivityLevel"] = user_summary["avg_steps"].apply(classify_user)
print("=== USER SEGMENTATION BY AVERAGE DAILY STEPS (33 USERS) ===")
print(user_summary["ActivityLevel"].value_counts(), "\n")
print("=== AVERAGE METRICS BY USER SEGMENT ===")
print(
    user_summary.groupby("ActivityLevel")[
        ["tracked_days", "avg_steps", "avg_sedentary_min", "avg_very_active_min"]
    ]
    .mean()
    .round(1)
)

=== DAILY ACTIVITY SUMMARY STATS (ALL 940 ROWS) ===
       TotalSteps  TotalDistance  VeryActiveMinutes  FairlyActiveMinutes  \
count      940.00         940.00             940.00               940.00   
mean      7637.91           5.49              21.16                13.56   
std       5087.15           3.92              32.84                19.99   
min          0.00           0.00               0.00                 0.00   
25%       3789.75           2.62               0.00                 0.00   
50%       7405.50           5.24               4.00                 6.00   
75%      10727.00           7.71              32.00                19.00   
max      36019.00          28.03             210.00               143.00   

       LightlyActiveMinutes  SedentaryMinutes  TotalActiveMinutes  Calories  
count                940.00            940.00              940.00    940.00  
mean                 192.81            991.21              227.54   2303.61  
std                  109.17  

In [4]:
# Export cleaned and aggregated datasets to /kaggle/working/ for Tableau Public
daily_activity_clean.to_csv("/kaggle/working/daily_activity_clean.csv", index=False)
sleep_day_clean.to_csv("/kaggle/working/sleep_day_clean.csv", index=False)
activity_sleep_merged.to_csv(
    "/kaggle/working/activity_sleep_merged.csv", index=False
)

# Aggregated hourly profile across all users
hourly_summary_export = (
    hourly_activity_clean.groupby("Hour", as_index=False)
    .agg(
        avg_steps=("StepTotal", "mean"),
        avg_total_intensity=("TotalIntensity", "mean"),
        avg_intensity=("AverageIntensity", "mean"),
    )
    .round(2)
)
hourly_summary_export.to_csv(
    "/kaggle/working/hourly_summary_export.csv", index=False
)

# User-level summary table with activity segmentation
user_summary_export = user_summary.reset_index()
user_summary_export.to_csv(
    "/kaggle/working/user_summary_export.csv", index=False
)

print("Exported all CSV files to /kaggle/working/ successfully!")

Exported all CSV files to /kaggle/working/ successfully!
